# Digital Crime Investigation
## Notebook 03 — Anomaly Detection & Case Investigation

**Author:** Baskara Kresna Juniarto  
**Project:** Transaction Fraud & Anomaly Analytics  

---
Phase 3: Apply the 7 anomaly indicators (A–G), compute composite
risk scores, and build evidence chains for named investigation cases.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_DIR = Path('../data')
IMG_DIR  = Path('../images')

txn      = pd.read_csv(DATA_DIR / 'transactions_clean.csv', parse_dates=['timestamp'])
users    = pd.read_csv(DATA_DIR / 'users.csv')
devices  = pd.read_csv(DATA_DIR / 'devices.csv', parse_dates=['first_seen','last_seen'])
merchants= pd.read_csv(DATA_DIR / 'merchants.csv')

txn_sorted = txn.sort_values(['user_id','timestamp']).reset_index(drop=True)
print(f'Investigation dataset: {len(txn_sorted):,} transactions')

## INDICATOR A — Impossible Travel
Consecutive transactions from different cities separated by < 60 minutes.

In [ ]:
t = txn_sorted.copy()
t['prev_location'] = t.groupby('user_id')['location_id'].shift(1)
t['prev_timestamp']= t.groupby('user_id')['timestamp'].shift(1)
t['prev_txn_id']   = t.groupby('user_id')['transaction_id'].shift(1)
t['gap_min']       = (t['timestamp'] - t['prev_timestamp']).dt.total_seconds() / 60

impossible_travel = t[
    t['prev_location'].notna() &
    (t['location_id'] != t['prev_location']) &
    (t['gap_min'] < 60)
][['user_id','prev_txn_id','transaction_id','prev_location',
   'location_id','prev_timestamp','timestamp','gap_min','amount','device_id']].copy()

impossible_travel.columns = ['user_id','from_txn','to_txn','from_city',
                              'to_city','from_time','to_time','gap_min','amount','device_id']

print(f'Impossible Travel events detected: {len(impossible_travel)}')
impossible_travel

## INDICATOR B — Transaction Burst
≥ 5 transactions by the same user within any 10-minute window.

In [ ]:
def detect_burst(group, window_min=10, threshold=5):
    group = group.sort_values('timestamp').reset_index(drop=True)
    burst_users = []
    for i, row in group.iterrows():
        window_end   = row['timestamp']
        window_start = window_end - pd.Timedelta(minutes=window_min)
        cnt = ((group['timestamp'] >= window_start) &
               (group['timestamp'] <= window_end)).sum()
        if cnt >= threshold:
            burst_users.append({'user_id': row['user_id'],
                                'window_center': row['timestamp'],
                                'txn_in_window': cnt})
    return pd.DataFrame(burst_users)

burst_results = []
for uid, grp in txn_sorted.groupby('user_id'):
    res = detect_burst(grp)
    if len(res) > 0:
        burst_results.append(res)

if burst_results:
    burst_df = pd.concat(burst_results).drop_duplicates()
    print(f'Burst users detected: {burst_df["user_id"].nunique()}')
    print(burst_df.groupby('user_id')['txn_in_window'].max().sort_values(ascending=False))
else:
    print('No burst events detected.')

## INDICATOR C — Refund Abuse
Users with refund rate ≥ 5× population average (min 5 transactions).

In [ ]:
pop_avg_refund = txn['refund_flag'].mean()
print(f'Population average refund rate: {pop_avg_refund*100:.2f}%')

user_refund = txn.groupby('user_id').agg(
    txn_count=('transaction_id','count'),
    refund_count=('refund_flag','sum'),
    refunded_amount=('amount', lambda x: x[txn.loc[x.index,'refund_flag']==1].sum()),
).reset_index()
user_refund['refund_rate'] = user_refund['refund_count'] / user_refund['txn_count']

refund_abusers = user_refund[
    (user_refund['txn_count'] >= 5) &
    (user_refund['refund_rate'] >= pop_avg_refund * 5)
].sort_values('refund_rate', ascending=False)

refund_abusers['multiple_of_avg'] = (refund_abusers['refund_rate'] / pop_avg_refund).round(1)
print(f'\nRefund abuse users: {len(refund_abusers)}')
refund_abusers

## INDICATOR D — Unusual Amount
Transactions exceeding 8× the user's own category average.

In [ ]:
user_cat_avg = txn.groupby(['user_id','category'])['amount'].agg(
    avg_amt='mean', cnt='count'
).reset_index()

txn_with_avg = txn.merge(user_cat_avg, on=['user_id','category'], how='left')
txn_with_avg['multiple'] = txn_with_avg['amount'] / txn_with_avg['avg_amt']

unusual_amount = txn_with_avg[
    (txn_with_avg['cnt'] >= 3) &
    (txn_with_avg['multiple'] >= 8)
][['transaction_id','user_id','timestamp','amount','category',
   'avg_amt','multiple','device_id','location_id','refund_flag']]\
    .sort_values('multiple', ascending=False)

print(f'Unusual amount transactions: {len(unusual_amount)}')
print(f'Users affected: {unusual_amount["user_id"].nunique()}')
unusual_amount

## INDICATOR E — New Device + High Value
First use of a device combined with transaction > p95.

In [ ]:
p95 = txn['amount'].quantile(0.95)
print(f'p95 threshold: Rp {p95:,.0f}')

device_first_use = txn.groupby(['user_id','device_id'])['timestamp'].min().reset_index()
device_first_use.columns = ['user_id','device_id','first_used_at']

txn_first = txn.merge(device_first_use, on=['user_id','device_id'])
new_device_high_val = txn_first[
    (txn_first['timestamp'] == txn_first['first_used_at']) &
    (txn_first['amount'] > p95)
][['transaction_id','user_id','device_id','timestamp','amount',
   'category','location_id','payment_method','refund_flag']]\
    .sort_values('amount', ascending=False)

print(f'New device + high-value events: {len(new_device_high_val)}')
new_device_high_val

## INDICATOR F — Shared Device
Single device_id used by ≥ 2 distinct user accounts.

In [ ]:
device_users = txn.groupby('device_id').agg(
    user_count=('user_id','nunique'),
    users=('user_id', lambda x: ', '.join(sorted(x.unique()))),
    txn_count=('transaction_id','count'),
    total_amount=('amount','sum'),
).reset_index()

shared_devices = device_users[device_users['user_count'] >= 2].sort_values(
    'user_count', ascending=False)

print(f'Shared devices detected: {len(shared_devices)}')
shared_devices.merge(devices[['device_id','device_type','operating_system']],
                     on='device_id')

## INDICATOR G — Unusual Location
Transaction outside home city with amount > user's own p75.

In [ ]:
user_p75 = txn.groupby('user_id')['amount'].quantile(0.75).reset_index()
user_p75.columns = ['user_id','p75_amt']

txn_geo = txn.merge(users[['user_id','city']], on='user_id')\
              .merge(user_p75, on='user_id')

unusual_location = txn_geo[
    (txn_geo['location_id'] != txn_geo['city']) &
    (txn_geo['amount'] > txn_geo['p75_amt'])
][['transaction_id','user_id','city','location_id','timestamp',
   'amount','p75_amt','device_id','payment_method','refund_flag']]
unusual_location['multiple'] = (unusual_location['amount'] / unusual_location['p75_amt']).round(1)
unusual_location = unusual_location.sort_values('multiple', ascending=False)

print(f'Unusual location transactions: {len(unusual_location)}')
print(f'Users affected: {unusual_location["user_id"].nunique()}')
unusual_location.head(15)

## Composite Risk Score
Each indicator contributes a weighted score. Sum determines risk level.

In [ ]:
WEIGHTS = {
    'impossible_travel'    : 35,
    'transaction_burst'    : 25,
    'refund_abuse'         : 20,
    'unusual_amount'       : 20,
    'new_device_high_value': 20,
    'shared_device'        : 15,
    'unusual_location'     : 10,
}

all_users = set(txn['user_id'].unique())
score_map = {u: {} for u in all_users}

for uid in impossible_travel['user_id'].unique():
    score_map[uid]['impossible_travel'] = WEIGHTS['impossible_travel']

if burst_results:
    for uid in burst_df['user_id'].unique():
        score_map[uid]['transaction_burst'] = WEIGHTS['transaction_burst']

for uid in refund_abusers['user_id'].unique():
    score_map[uid]['refund_abuse'] = WEIGHTS['refund_abuse']

for uid in unusual_amount['user_id'].unique():
    score_map[uid]['unusual_amount'] = WEIGHTS['unusual_amount']

for uid in new_device_high_val['user_id'].unique():
    score_map[uid]['new_device_high_value'] = WEIGHTS['new_device_high_value']

shared_user_list = []
for u_str in shared_devices['users'].str.split(', '):
    shared_user_list.extend(u_str)
for uid in set(shared_user_list):
    if uid in score_map:
        score_map[uid]['shared_device'] = WEIGHTS['shared_device']

for uid in unusual_location['user_id'].unique():
    score_map[uid]['unusual_location'] = WEIGHTS['unusual_location']

risk_rows = []
for uid, indicators in score_map.items():
    total = sum(indicators.values())
    if total > 0:
        risk_rows.append({
            'user_id': uid,
            'composite_score': total,
            'indicators': ' | '.join(sorted(indicators.keys(), key=lambda k: -WEIGHTS[k])),
            'indicator_count': len(indicators),
        })

risk_df = pd.DataFrame(risk_rows).sort_values('composite_score', ascending=False)
risk_df['risk_level'] = pd.cut(
    risk_df['composite_score'],
    bins=[0, 30, 60, 9999],
    labels=['LOW', 'MEDIUM', 'HIGH']
)

print(f'Users with risk signals: {len(risk_df)}')
print(risk_df['risk_level'].value_counts())
risk_df.head(20)

## Risk Distribution Visualisation

In [ ]:
level_counts = risk_df['risk_level'].value_counts().reindex(['HIGH','MEDIUM','LOW'])
level_colors = {'HIGH': 'crimson', 'MEDIUM': 'darkorange', 'LOW': 'steelblue'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart — risk level counts
bars = axes[0].bar(level_counts.index, level_counts.values,
                   color=[level_colors[l] for l in level_counts.index])
for bar, val in zip(bars, level_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 str(val), ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Flagged Users by Risk Level', fontweight='bold')
axes[0].set_ylabel('User Count')

# Scatter — score vs indicator count
color_list = risk_df['risk_level'].map(level_colors)
axes[1].scatter(risk_df['indicator_count'], risk_df['composite_score'],
                c=color_list, s=80, alpha=0.7, edgecolors='white')
axes[1].set_title('Risk Score vs Indicator Count', fontweight='bold')
axes[1].set_xlabel('Number of Triggered Indicators')
axes[1].set_ylabel('Composite Risk Score')
axes[1].axhline(y=61, color='crimson',    linestyle='--', alpha=0.5, label='HIGH threshold')
axes[1].axhline(y=31, color='darkorange', linestyle='--', alpha=0.5, label='MEDIUM threshold')
axes[1].legend()

# Legend patches
patches = [mpatches.Patch(color=c, label=l) for l, c in level_colors.items()]
axes[1].legend(handles=patches + axes[1].get_legend_handles_labels()[0],
               labels=[p.get_label() for p in patches] + axes[1].get_legend_handles_labels()[1])

plt.tight_layout()
plt.savefig(IMG_DIR / '03_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Deep-Dive: CASE-001 — U0023 Impossible Travel

In [ ]:
case_u0023 = txn[txn['user_id'] == 'U0023'].sort_values('timestamp')\
               .merge(merchants[['merchant_id','merchant_name']], on='merchant_id')\
               .merge(devices[['device_id','device_type','operating_system']], on='device_id')

case_u0023['gap_min'] = case_u0023['timestamp'].diff().dt.total_seconds() / 60

print('=== CASE-001: U0023 — Evidence Chain ===')
print(case_u0023[['transaction_id','timestamp','location_id','amount',
                  'device_id','device_type','merchant_name','payment_method',
                  'refund_flag','gap_min']].to_string(index=False))

## Deep-Dive: CASE-004 — U0045 Transaction Burst

In [ ]:
burst_window = txn[
    (txn['user_id'] == 'U0045') &
    (txn['timestamp'] >= '2024-02-14 01:00:00') &
    (txn['timestamp'] <= '2024-02-14 01:10:00')
].sort_values('timestamp')

print('=== CASE-004: U0045 — Burst Window ===')
print(burst_window[['transaction_id','timestamp','amount','merchant_id',
                    'category','payment_method']].to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(burst_window['timestamp'], burst_window['amount'],
           s=200, color='crimson', zorder=5)
ax.plot(burst_window['timestamp'], burst_window['amount'],
        color='crimson', linewidth=1.5, linestyle='--', alpha=0.6)
ax.set_title('U0045 — Transaction Burst (6 × Rp500K in 6 minutes)', fontweight='bold')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Amount (IDR)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'Rp{x/1e3:.0f}K'))
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(IMG_DIR / '03_case004_burst.png', dpi=150, bbox_inches='tight')
plt.show()

## Investigation Summary — Case Register

In [ ]:
case_register = pd.DataFrame([
    {'Case ID':'CASE-001','Type':'Impossible Travel',    'Users':'U0023',       'Risk':'HIGH'},
    {'Case ID':'CASE-002','Type':'Impossible Travel',    'Users':'U0067',       'Risk':'HIGH'},
    {'Case ID':'CASE-003','Type':'Impossible Travel',    'Users':'U0112',       'Risk':'HIGH'},
    {'Case ID':'CASE-004','Type':'Transaction Burst',    'Users':'U0045, U0089','Risk':'HIGH'},
    {'Case ID':'CASE-005','Type':'Refund Abuse',         'Users':'U0034, U0078, U0156','Risk':'HIGH'},
    {'Case ID':'CASE-006','Type':'Unusual Amount',       'Users':'U0019, U0057, U0103, U0144, U0191','Risk':'MEDIUM'},
    {'Case ID':'CASE-007','Type':'New Device High Value','Users':'U0033, U0071, U0118, U0162','Risk':'MEDIUM'},
    {'Case ID':'CASE-008','Type':'Shared Device',        'Users':'D0150 (4 accts), D0151 (5 accts)','Risk':'MEDIUM'},
])

print('=== INVESTIGATION CASE REGISTER ===')
print(case_register.to_string(index=False))

# Export risk scores
risk_df.to_csv(DATA_DIR / 'risk_scores.csv', index=False)
print('\nrisk_scores.csv exported ✓')